# Apply Work-Author Curations

Lays work-author curations over `openalex.works.work_authors` in the view `openalex.works.work_authors_curated`. Runs after `Author_Matching` and before `CreateWorkAuthorships`, which takes every seat's `author_id` from the view.

## Curations are a view, not a write (oxjob #1438, 2026-09-30)

The matcher (`MatchAuthors`) owns `work_authors`; nothing here writes to it. The view recomputes the effective `author_id` of every seat from tonight's curation tables, so a curation that exists tonight is applied tonight and one that is gone tomorrow (deleted by its user, or with its account, oxjob #1436) is not applied tomorrow. The matcher's own answer stays in the table underneath: deleting a claim returns the seat to it, and deleting a removal brings the disclaimed author back.

Until 2026-09-30 this notebook MERGEd curations into `work_authors` in place, which made them permanent: the matcher never revisits a bound seat on an old work, so a deleted curation stayed applied. Seats curated before the switch keep those values as their matcher value (accepted, oxjob #1438). A disclaimed seat keeps the matcher's author in the table and is served NULL, so the matcher does not rematch it.

## Semantics (unchanged from the old MERGEs)

- **Claim**: the latest claim per `(work_id, raw_author_name)` (`created DESC, curation_id DESC`; conflicting claims on one slot are real) sets `author_id`. Claims to deleted profiles are skipped (oxjob #1354).
- **Remove**: after claims, any seat whose `author_id` is disclaimed on that work is served NULL, unless a claim of the same work by the same author is newer than the latest removal (oxjob #1430, 2026-09-30: the person re-claimed the paper after removing it).

## Curations follow merged works (oxjob #2008, 2026-11-04)

A curation names the work id the curator saw. When that work is later merged into another (`MergeIdenticalKeyWorks`, `MergeDeclaredLinks`), the curation now lands on the winner: the view `openalex.works.work_author_curations_resolved` gives each curation a `resolved_work_id` through `openalex.works.merged_work_ids_map` (oxjob #2006: the live winner, chains followed; a loser that is live again keeps its own curations), and `via_merge` marks the carried-over ones. Before this, merges had left 1,547,893 claims on 685,762 dead works (3 Nov 2026), 65,464 of them served nowhere. The apply receipt (oxjob #2007) reads the same view.

Curations made on the work itself outrank carried-over ones, so every seat such a curation decides is served exactly as before; carried-over curations change only seats the matcher chose:
- a carried-over claim fills a slot only when no claim made on the work holds it, and never takes a seat from an author that a claim on the work names;
- a carried-over removal never beats a claim made on the work; a carried-over claim never beats a removal made on the work;
- among carried-over curations, the latest wins, as everywhere else.

A carried-over claim whose name is not on the winner's byline applies nowhere (the receipt reports it as `target_gone`).

## Propagation

`CreateWorkAuthorships` rebuilds every work's authorships each run (oxjob #660) from the view, so curations and their deletions reach `work_authorships` the same night.


## The views

`work_author_curations_resolved_snapshot` (a table, rebuilt here each night): every claim and removal, one row per curation, with the work it lands on tonight; for a carried-over claim, `direct_author_ids` lists the authors that claims made on that work name. `work_author_curations_resolved` is the same rows as a view, for the apply receipt (oxjob #2007). It is a table so the merge lookup stays out of `CreateWorkAuthorships`' plan (night 2026-11-05: inlined as a view, it slowed that statement past its 40-min timeout).

`work_authors_curated`: every `work_authors` column, `author_id` replaced by the curated value. `DISTINCT` on the claim side is not enough because conflicting claims carry different `author_id`s, hence the latest-wins window; the removal side collapses fan-in with `DISTINCT` because every colliding row means the same thing. Both joins are on keys that are unique after those steps, so the view has exactly one row per `work_authors` row.

In [ ]:
%sql
-- oxjob #2008: one row per work-author curation, with the work it lands on tonight. A curation names the work id the
-- curator saw; when that work is later merged away, merged_work_ids_map (oxjob #2006: the live winner, chains followed;
-- a loser that is live again is left out) names the work it now lives on. A TABLE, not a view (5 Nov 2026): as a view its
-- joins to the 60M-row map were inlined into CreateWorkAuthorships' main statement, whose plan then ran on ~59 tasks
-- instead of ~390 and hit the 40-min timeout (night 2026-11-05). It reads only the curation tables and the map (not
-- work_authors or deleted_authors), so building it here gives Authorships, the receipt and DeleteEmptiedAuthors one answer.
CREATE OR REPLACE TABLE openalex.works.work_author_curations_resolved_snapshot
COMMENT 'Every work-author claim and removal with resolved_work_id: its work_id, or the live work it was merged into (merged_work_ids_map); via_merge marks the carried-over ones; direct_author_ids (carried-over claims only) names the authors that claims made on that work name. Built each night by ApplyWorkAuthorCurations so work_authors_curated reads a table, not joins (oxjob #2008).'
AS
WITH resolved AS (
  SELECT c.curation_id, 'claim' AS kind, c.user_id, c.author_id, c.raw_author_name, c.created, c.work_id,
         COALESCE(m.merge_into_id, c.work_id) AS resolved_work_id, m.id IS NOT NULL AS via_merge
  FROM openalex.works.work_author_claim_curations c
  LEFT JOIN openalex.works.merged_work_ids_map m ON m.id = c.work_id
  UNION ALL
  SELECT r.curation_id, 'remove' AS kind, r.user_id, r.author_id, CAST(NULL AS STRING) AS raw_author_name, r.created, r.work_id,
         COALESCE(m.merge_into_id, r.work_id) AS resolved_work_id, m.id IS NOT NULL AS via_merge
  FROM openalex.works.work_author_remove_curations r
  LEFT JOIN openalex.works.merged_work_ids_map m ON m.id = r.work_id
),
-- authors that a claim made on the work itself names, on works a carried-over claim reaches
direct AS (
  SELECT resolved_work_id, COLLECT_SET(author_id) AS direct_author_ids
  FROM resolved
  WHERE kind = 'claim' AND NOT via_merge
    AND resolved_work_id IN (SELECT resolved_work_id FROM resolved WHERE kind = 'claim' AND via_merge)
  GROUP BY resolved_work_id
)
SELECT r.*, CASE WHEN r.kind = 'claim' AND r.via_merge THEN d.direct_author_ids END AS direct_author_ids
FROM resolved r
LEFT JOIN direct d ON d.resolved_work_id = r.resolved_work_id;

-- The view the apply receipt (oxjob #2007) and the Verify cells read: the same rows, without the helper column.
CREATE OR REPLACE VIEW openalex.works.work_author_curations_resolved
COMMENT 'Every work-author claim and removal with resolved_work_id: its work_id, or the live work it was merged into (merged_work_ids_map); via_merge marks the carried-over ones (ApplyWorkAuthorCurations, oxjob #2008). Reads work_author_curations_resolved_snapshot, rebuilt each night.'
AS
SELECT curation_id, kind, user_id, author_id, raw_author_name, created, work_id, resolved_work_id, via_merge
FROM openalex.works.work_author_curations_resolved_snapshot;

CREATE OR REPLACE VIEW openalex.works.work_authors_curated
COMMENT 'work_authors with the current work-author curations laid on top (ApplyWorkAuthorCurations, oxjob #1438; curations on merged-away works follow them, oxjob #2008). Read this, not work_authors, for the served author_id of a seat.'
AS
WITH claims AS (
  SELECT c.resolved_work_id AS work_id, c.raw_author_name, c.author_id, c.via_merge, c.direct_author_ids
  FROM openalex.works.work_author_curations_resolved_snapshot c
  -- never bind a seat to a deleted profile (oxjob #1354)
  LEFT ANTI JOIN openalex.authors.deleted_authors d ON d.author_id = c.author_id
  WHERE c.kind = 'claim'
  -- latest claim per slot; a claim made on the work itself outranks one carried over from a merged-away work (oxjob #2008)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.resolved_work_id, c.raw_author_name
    ORDER BY c.via_merge, c.created DESC, c.curation_id DESC
  ) = 1
),
removals AS (
  SELECT DISTINCT r.work_id, r.author_id
  FROM (
    SELECT resolved_work_id AS work_id, author_id, via_merge, MAX(created) AS created
    FROM openalex.works.work_author_curations_resolved_snapshot
    WHERE kind = 'remove'
    GROUP BY resolved_work_id, author_id, via_merge
  ) r
  -- a claim newer than the latest removal of the same (work, author) wins (oxjob #1430). Curations made on the work
  -- itself outrank carried-over ones (oxjob #2008): a direct claim beats a carried-over removal at any age, and a
  -- carried-over claim never beats a direct removal.
  WHERE NOT EXISTS (
    SELECT 1 FROM openalex.works.work_author_curations_resolved_snapshot c
    WHERE c.kind = 'claim' AND c.resolved_work_id = r.work_id AND c.author_id = r.author_id
      AND ((c.created > r.created AND (NOT c.via_merge OR r.via_merge)) OR (NOT c.via_merge AND r.via_merge))
  )
)
SELECT
  wa.work_id,
  wa.author_sequence,
  CASE WHEN rm.author_id IS NOT NULL THEN NULL
       -- a carried-over claim never takes a seat from an author that a claim made on the work names (oxjob #2008)
       WHEN c.via_merge AND ARRAY_CONTAINS(c.direct_author_ids, wa.author_id) THEN wa.author_id
       ELSE COALESCE(c.author_id, wa.author_id) END AS author_id,
  wa.raw_author_name,
  wa.raw_affiliation_strings,
  wa.is_corresponding,
  wa.created_at,
  wa.updated_at
FROM openalex.works.work_authors wa
LEFT JOIN claims c
  ON c.work_id = wa.work_id AND c.raw_author_name = wa.raw_author_name
LEFT JOIN removals rm
  ON rm.work_id = wa.work_id
 AND rm.author_id = CASE WHEN c.via_merge AND ARRAY_CONTAINS(c.direct_author_ids, wa.author_id) THEN wa.author_id
                         ELSE COALESCE(c.author_id, wa.author_id) END

## Restore profiles a seat serves again (oxjob #1949)

`DeleteEmptiedAuthors` deletes a profile whose every live seat is curated away (ledger reason `empty_curated_away`): a claim
serves the seat to another profile, or a removal serves it to no one. The matcher's value stays underneath in `work_authors`
(`UpdateWorkAuthors` Step 2a leaves those seats alone), so when the curation goes (deleted by its user, with its account, or
in a bulk cancel such as #1908's), the view serves the deleted id again. Here, before `CreateWorkAuthorships` reads the view,
such a profile comes back from its ledger row (the profile row verbatim, the #1354 undo): any profile ledgered
`empty_curated_away` that the view serves on a work in `openalex_works_base`, or that a claim targets (the view skips claims
to deleted profiles, so restoring it here lets the claim apply tonight). Its ledger row goes; `restored_authors` keeps a
copy. `Create_Authors` re-dates it in the morning (content hash), so the authors sync re-indexes it.

The same goes for a profile ledgered `empty_seats_not_on_byline` (oxjob #1949 A3): its seats sat on works whose current
byline had no author at their position. A seat counts as served only when the view binds it **and** the work's byline in
`openalex_works_base` has that position (`CreateWorkAuthorships` binds ids by position), for both reasons: a profile can hold
seats of both kinds, and a seat past the byline's end serves no one whatever the view says.

Sized 1 Nov 2026: the 27 Oct bulk cancel (288,486 bot claims) would have restored 3,718 profiles, #1908's 30 Oct cancel
(117,562) 1,077; single cancels run 0-11 a night.


In [ ]:
%sql
-- oxjob #1949: restore `empty_curated_away` and `empty_seats_not_on_byline` profiles that a seat serves again or a claim
-- targets (see the cell above).
CREATE TABLE IF NOT EXISTS openalex.authors.restored_authors (
  author_id BIGINT, display_name STRING, full_name STRING, orcid STRING, created_date TIMESTAMP, updated_date TIMESTAMP,
  reason STRING, deleted_at TIMESTAMP, restored_at TIMESTAMP)
COMMENT 'oxjob #1949: profiles restored from openalex.authors.deleted_authors because a seat or a claim pointed to them again (ApplyWorkAuthorCurations)';

CREATE OR REPLACE TABLE openalex.authors.curated_away_restore_batch AS
WITH a2 AS (SELECT author_id FROM openalex.authors.deleted_authors WHERE reason IN ('empty_curated_away', 'empty_seats_not_on_byline')),
base_seats AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id
  FROM openalex.works.work_authors wa
  JOIN a2 ON a2.author_id = wa.author_id
  JOIN openalex.works.openalex_works_base b ON b.id = wa.work_id
  -- the byline has this seat: past its end the seat serves no one (oxjob #1949 A3)
  WHERE COALESCE(SIZE(b.authorships), 0) > wa.author_sequence)
SELECT DISTINCT s.author_id
FROM base_seats s
JOIN openalex.works.work_authors_curated v
  ON v.work_id = s.work_id AND v.author_sequence = s.author_sequence AND v.author_id = s.author_id
UNION
SELECT DISTINCT c.author_id
FROM openalex.works.work_author_claim_curations c JOIN a2 ON a2.author_id = c.author_id;

INSERT INTO openalex.authors.restored_authors
SELECT d.author_id, d.display_name, d.full_name, d.orcid, d.created_date, d.updated_date, d.reason, d.deleted_at, current_timestamp()
FROM openalex.authors.deleted_authors d
JOIN openalex.authors.curated_away_restore_batch r ON r.author_id = d.author_id
WHERE d.reason IN ('empty_curated_away', 'empty_seats_not_on_byline');

INSERT INTO openalex.authors.authors (id, display_name, full_name, orcid, created_date, updated_date)
SELECT d.author_id, d.display_name, d.full_name, d.orcid, d.created_date, d.updated_date
FROM openalex.authors.deleted_authors d
JOIN openalex.authors.curated_away_restore_batch r ON r.author_id = d.author_id
LEFT ANTI JOIN openalex.authors.authors a ON a.id = d.author_id
WHERE d.reason IN ('empty_curated_away', 'empty_seats_not_on_byline');

DELETE FROM openalex.authors.deleted_authors
WHERE reason IN ('empty_curated_away', 'empty_seats_not_on_byline')
  AND author_id IN (SELECT author_id FROM openalex.authors.curated_away_restore_batch);

SELECT (SELECT COUNT(*) FROM openalex.authors.curated_away_restore_batch) AS restored_tonight,
       (SELECT COUNT(*) FROM openalex.authors.deleted_authors WHERE reason = 'empty_curated_away') AS still_deleted_curated_away,
       (SELECT COUNT(*) FROM openalex.authors.deleted_authors WHERE reason = 'empty_seats_not_on_byline') AS still_deleted_not_on_byline;


## Verify

Seats where the view differs from the table are the seats curations change tonight. Only seats on curated works can differ, so both cells look at those works only. Both counts are totals (the view is rebuilt from scratch every read).


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM openalex.works.work_author_claim_curations)   AS claim_curations_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations) AS removal_curations_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_curations_resolved WHERE via_merge) AS curations_via_merge,
    COUNT(CASE WHEN v.author_id IS DISTINCT FROM wa.author_id THEN 1 END) AS seats_changed_by_curations,
    COUNT(CASE WHEN v.author_id IS NOT NULL AND v.author_id IS DISTINCT FROM wa.author_id THEN 1 END) AS seats_claimed,
    COUNT(CASE WHEN v.author_id IS NULL AND wa.author_id IS NOT NULL THEN 1 END) AS seats_removed
FROM openalex.works.work_authors wa
JOIN openalex.works.work_authors_curated v
  ON v.work_id = wa.work_id AND v.author_sequence = wa.author_sequence
WHERE wa.work_id IN (
    SELECT resolved_work_id FROM openalex.works.work_author_curations_resolved
)


In [ ]:
%sql
-- Spot-check: seats the curations change tonight, table value beside served value.
SELECT wa.work_id, wa.author_sequence, wa.raw_author_name,
       wa.author_id AS matcher_author_id, v.author_id AS served_author_id
FROM openalex.works.work_authors wa
JOIN openalex.works.work_authors_curated v
  ON v.work_id = wa.work_id AND v.author_sequence = wa.author_sequence
WHERE v.author_id IS DISTINCT FROM wa.author_id
  AND wa.work_id IN (
    SELECT resolved_work_id FROM openalex.works.work_author_curations_resolved
)
ORDER BY wa.work_id
LIMIT 100
